# 03 — LLM Inference with Gemini API

Full walkthrough of calling the Gemini API, setting parameters, streaming responses, and understanding costs.

## Setup

Before running this notebook, set your OpenAI API key:

```bash
export OPENAI_API_KEY="your-api-key-here"
```

In [ ]:
# Install Google GenAI SDK if not installed
# !pip install google-genai

import openai

import os
import time

# Initialize client
client = openai.OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY", "your-api-key-here")
)

print("Gemini client initialized.")

## 1. Basic API Call

The simplest way to call the Gemini API.

In [ ]:
# Simple generation
response = client.models.generate_content(
    model="gpt-4o-mini",
    contents="What is the capital of France?"
)

print("Response:")
print(response.choices[0].message.content)

In [ ]:
# Examine the response object
print("Usage metadata:")
print(f"  Prompt tokens: {response.usage.prompt_token_count}")
print(f"  Output tokens: {response.usage.candidates_token_count}")
print(f"  Total tokens: {response.usage.total_token_count}")

## 2. Multi-turn Conversation

In [ ]:
# Multi-turn conversation using Content objects
conversation = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": [
        types.Content(
            role="user"}],
            parts=[types.Part.from_text("Hi, my name is Alice.")]
        ),
        types.Content(
            role="model",
            parts=[types.Part.from_text("Hello Alice! Nice to meet you.")]
        ),
        types.Content(
            role="user",
            parts=[types.Part.from_text("What was my name again?")]
        ),
    ]
)

print(conversation.text)

## 3. System Instructions

System instructions set the model's behavior and persona.

In [ ]:
# Using system instructions
response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": "Explain quantum computing in 3 sentences."}],
    
        system_instruction="You are a physics professor who explains complex topics in simple, accessible language."
    
)

print(response.choices[0].message.content)

## 4. Temperature: Controlling Randomness

Temperature controls how creative vs. deterministic the output is.

In [ ]:
# Compare different temperatures
prompt = "Write a creative opening line for a story about a robot."

temperatures = [0.0, 0.5, 1.0, 2.0]

for temp in temperatures:
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}],
        
            temperature=temp,
            max_output_tokens=100
        
    )
    print(f"Temperature {temp}:")
    print(f"  {response.choices[0].message.content[:100]}...")
    print()

## 5. Top-p (Nucleus Sampling)

Top-p limits the token pool to the most probable tokens whose cumulative probability exceeds p.

In [ ]:
# Compare different top_p values
prompt = "Complete this sentence: The future of AI is"

top_p_values = [0.1, 0.5, 0.9, 1.0]

for top_p in top_p_values:
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}],
        
            temperature=0.8,
            top_p=top_p,
            max_output_tokens=50
        
    )
    print(f"Top-p {top_p}: {response.choices[0].message.content}")

## 6. Streaming Responses

Stream tokens as they're generated instead of waiting for the full response.

In [ ]:
# Streaming response
response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": "Explain the concept of recursion in programming in 3 sentences."}],
    
        max_output_tokens=200,
        temperature=0.7,
        stream=True
    
)

print("Streaming response:")
print("-" * 40)
for chunk in response:
    if chunk.text:
        print(chunk.text, end="", flush=True)
print("\n")

## 7. Structured Output

Force the model to output JSON or other structured formats.

In [ ]:
# Get structured JSON output
response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": "List 3 programming languages with their primary use cases."}],
    
        response_mime_type="application/json",
        temperature=0.3
    
)

print("JSON response:")
print(response.choices[0].message.content)

In [ ]:
# Parse the JSON
import json
data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2))

## 8. Understanding Token Usage and Cost

Track token usage across multiple calls.

In [ ]:
# Track token usage
def make_request(prompt, model="gpt-4o-mini", **kwargs):
    """Make a request and return response with token counts."""
    start_time = time.time()
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        **kwargs
    )
    elapsed = time.time() - start_time
    
    return {
        "text": response.choices[0].message.content,
        "input_tokens": response.usage.prompt_token_count,
        "output_tokens": response.usage.candidates_token_count,
        "total_tokens": response.usage.total_token_count,
        "elapsed": elapsed
    }

In [ ]:
# Test with different prompt lengths
prompts = [
    "What is 2+2?",
    "Explain what a variable is in programming.",
    "Write a detailed explanation of how neural networks work, including forward propagation, backpropagation, and gradient descent. Include examples.",
]

print(f"{'Prompt (first 50 chars)':<55} {'In Tokens':<10} {'Out Tokens':<12} {'Time (s)':<10}")
print("-" * 90)

for prompt in prompts:
    result = make_request(prompt, max_output_tokens=200, temperature=0.7)
    print(f"{prompt[:50]:<55} {result['input_tokens']:<10} {result['output_tokens']:<12} {result['elapsed']:<10.2f}")

## 9. Comparing Model Parameters

Experiment with different parameter combinations.

In [ ]:
# Systematic comparison of temperature and top_p
prompt = "Write a haiku about programming."

configs = [
    {"temperature": 0.0, "top_p": 1.0, "label": "temp=0.0, top_p=1.0"},
    {"temperature": 0.5, "top_p": 0.9, "label": "temp=0.5, top_p=0.9"},
    {"temperature": 1.0, "top_p": 0.9, "label": "temp=1.0, top_p=0.9"},
    {"temperature": 1.5, "top_p": 0.8, "label": "temp=1.5, top_p=0.8"},
    {"temperature": 2.0, "top_p": 0.5, "label": "temp=2.0, top_p=0.5"},
]

for config in configs:
    label = config.pop("label")
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}],
        
            **config,
            max_output_tokens=50
        
    )
    print(f"{label}:")
    print(f"  {response.choices[0].message.content}")
    print()

## 10. Error Handling

Handle common API errors gracefully.

In [ ]:
import openai

def safe_generate(prompt, model="gpt-4o-mini", **kwargs):
    """Generate content with error handling."""
    try:
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            **kwargs
        )
        return {"success": True, "text": response.choices[0].message.content}
    except exceptions.ResourceExhausted:
        return {"success": False, "error": "Rate limit exceeded. Try again later."}
    except exceptions.InvalidArgument as e:
        return {"success": False, "error": f"Invalid argument: {e}"}
    except Exception as e:
        return {"success": False, "error": f"Unexpected error: {e}"}

# Test error handling
result = safe_generate("Hello")
print(result)

## Summary

| Parameter | Effect | Typical Value |
|-----------|--------|---------------|
| temperature | Controls randomness (0=deterministic, 2=very random) | 0.0-1.0 |
| top_p | Limits token pool by cumulative probability | 0.8-0.95 |
| max_output_tokens | Maximum response length | 100-4096 |
| stream | Return tokens as generated | True/False |

**Best practices:**
- Use low temperature (0.0-0.3) for factual/precise tasks
- Use medium temperature (0.5-0.8) for balanced tasks
- Use high temperature (0.8-1.2) for creative tasks
- Monitor token usage for cost control

## Exercises

1. Create a function that runs the same prompt 5 times with temperature=1.0 and counts how many unique responses you get.
2. Build a simple chatbot that maintains conversation history and tracks total token usage.
3. Experiment with system instructions to make the model respond in different styles (formal, casual, poetic).